# LangGraph 실습 

- 상태 기반 워크플로와 에이전트

## Step 01 · 환경 설정

In [ ]:
import operator
import os
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
print("OpenAI:", "OK" if os.getenv("OPENAI_API_KEY") else "없음")

llm = ChatOpenAI(model="gpt-6-luna", temperature=0, reasoning={"effort": "none"})

## Step 02 · 첫 그래프

-  State · Node · Edge

In [ ]:
# 주문 금액을 받아 할인 → 배송비 → 최종 금액을 계산하는 LLM 없는 그래프

In [ ]:
class OrderState(TypedDict):
    amount: int          # 주문 금액
    grade: str           # 회원 등급
    discount: int
    shipping: int
    total: int

In [ ]:
def apply_discount(state: OrderState) -> dict:
    rate = {"vip": 0.1, "gold": 0.05}.get(state["grade"], 0)
    return {"discount": int(state["amount"] * rate)}          # 바꿀 키만 반환

def add_shipping(state: OrderState) -> dict:
    paid = state["amount"] - state["discount"]
    return {"shipping": 0 if paid >= 50_000 else 3_000}

def summarize(state: OrderState) -> dict:
    return {"total": state["amount"] - state["discount"] + state["shipping"]}

In [ ]:
from langgraph.graph import END, START, StateGraph

builder = StateGraph(OrderState)
builder.add_node("discount", apply_discount)
builder.add_node("shipping", add_shipping)
builder.add_node("summary", summarize)
builder.add_edge(START, "discount")
builder.add_edge("discount", "shipping")
builder.add_edge("shipping", "summary")
builder.add_edge("summary", END)
order_graph = builder.compile()

In [ ]:
print(order_graph.invoke({"amount": 52_000, "grade": "gold"}))

In [ ]:
print(order_graph.invoke({"amount": 30_000, "grade": "basic"}))

In [ ]:
# 그래프 구조 확인 — Mermaid 텍스트 (https://mermaid.live 에 붙여 넣으면 그림으로 확인)
print(order_graph.get_graph().draw_mermaid())
# Jupyter에서는 이미지로 바로 표시
# from IPython.display import Image, display
# display(Image(order_graph.get_graph().draw_mermaid_png()))

## Step 03 · State와 Reducer

In [ ]:
# 같은 키를 여러 노드가 갱신할 때, 값을 **덮어쓸지 · 누적할지**를 Reducer가 정한다.

In [ ]:
class LogState(TypedDict):
    status: str                                  # Reducer 없음 → 덮어쓰기
    logs: Annotated[list[str], operator.add]     # operator.add → 리스트 누적

In [ ]:
def step_a(state: LogState):
    return {"status": "A 완료", "logs": ["A 실행"]}

def step_b(state: LogState):
    return {"status": "B 완료", "logs": ["B 실행"]}

In [ ]:
g = StateGraph(LogState)
g.add_node("a", step_a)
g.add_node("b", step_b)
g.add_edge(START, "a")
g.add_edge("a", "b")
g.add_edge("b", END)
print(g.compile().invoke({"status": "시작", "logs": ["입력"]}))
# >> {'status': 'B 완료', 'logs': ['입력', 'A 실행', 'B 실행']}

In [ ]:
# add_messages — 메시지 누적 + 같은 id면 교체, dict/튜플도 메시지로 변환
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

old = [HumanMessage("안녕", id="1"), AIMessage("반가워요", id="2")]
print(add_messages(old, [("user", "날씨 알려줘")]))                 # 추가
print(add_messages(old, [AIMessage("반갑습니다!", id="2")])[-1])     # 같은 id → 교체

In [ ]:
# MessagesState — messages 키(add_messages)가 미리 정의된 State. 필드를 추가해 확장한다.
from langgraph.graph import MessagesState

class ChatState(MessagesState):
    user_name: str

In [ ]:
def chatbot(state: ChatState):
    system = ("system", f"사용자 이름은 {state['user_name']}입니다. 이름을 불러 주며 짧게 답하세요.")
    return {"messages": [llm.invoke([system] + state["messages"])]}

In [ ]:
chat_graph = (
    StateGraph(ChatState)
    .add_node("chatbot", chatbot)
    .add_edge(START, "chatbot")
    .add_edge("chatbot", END)
    .compile()
)

In [ ]:
out = chat_graph.invoke({"messages": [("user", "LangGraph를 한 문장으로 설명해줘")], "user_name": "지수"})
for m in out["messages"]:
    print(f"[{m.type}] {m.content}")

## Step 04 · 조건부 엣지와 루프

In [ ]:
# (1) LLM 분류 결과로 분기 — 고객 문의 라우터
from pydantic import BaseModel, Field

class Route(BaseModel):
    category: Literal["refund", "delivery", "other"] = Field(description="문의 유형")

class InquiryState(TypedDict):
    question: str
    category: str
    answer: str

In [ ]:
classifier = llm.with_structured_output(Route)

def classify(state: InquiryState):
    r = classifier.invoke(f"고객 문의를 refund / delivery / other 중 하나로 분류하세요.\n문의: {state['question']}")
    return {"category": r.category}

def refund_node(state: InquiryState):
    return {"answer": "환불은 수령 후 7일 이내 마이페이지 > 주문 내역에서 신청할 수 있습니다."}

def delivery_node(state: InquiryState):
    return {"answer": "배송 조회는 주문 내역의 송장 번호로 확인할 수 있습니다."}

def other_node(state: InquiryState):
    return {"answer": llm.invoke(f"쇼핑몰 상담원으로서 두 문장으로 답하세요: {state['question']}").content}

def route_by_category(state: InquiryState) -> Literal["refund", "delivery", "other"]:
    return state["category"]

In [ ]:
b = StateGraph(InquiryState)
b.add_node("classify", classify)
b.add_node("refund", refund_node)
b.add_node("delivery", delivery_node)
b.add_node("other", other_node)
b.add_edge(START, "classify")
b.add_conditional_edges("classify", route_by_category)   # 반환값 = 다음 노드 이름
for n in ["refund", "delivery", "other"]:
    b.add_edge(n, END)
router = b.compile()

In [ ]:
for q in ["어제 받은 신발 환불하고 싶어요", "택배가 아직 안 왔어요", "포인트는 언제 적립돼요?"]:
    r = router.invoke({"question": q})
    print(f"[{r['category']}] {q} → {r['answer']}")

In [ ]:
# (2) 루프 — 작성 → 검토 → (기준 미달이면) 재작성, 최대 3회
class Review(BaseModel):
    score: int = Field(description="1~10점")
    feedback: str = Field(description="개선할 점 한 문장")

class WriteState(TypedDict):
    topic: str
    draft: str
    score: int
    feedback: str
    revision: int

In [ ]:
def write(state: WriteState):
    fb = f"\n이전 피드백: {state['feedback']}" if state.get("feedback") else ""
    draft = llm.invoke(f"'{state['topic']}' 제품 소개 문구를 40자 이내로 작성하세요.{fb}").content
    return {"draft": draft, "revision": state.get("revision", 0) + 1}

def review(state: WriteState):
    r = llm.with_structured_output(Review).invoke(
        f"다음 광고 문구를 명확성 · 구체성 기준으로 채점하세요.\n문구: {state['draft']}")
    return {"score": r.score, "feedback": r.feedback}

def should_rewrite(state: WriteState) -> Literal["write", "__end__"]:
    if state["score"] >= 8 or state["revision"] >= 3:
        return END
    return "write"

In [ ]:
wb = StateGraph(WriteState)
wb.add_node("write", write)
wb.add_node("review", review)
wb.add_edge(START, "write")
wb.add_edge("write", "review")
wb.add_conditional_edges("review", should_rewrite)     # review → write 로 되돌아가는 사이클
writer_graph = wb.compile()

In [ ]:
for step in writer_graph.stream({"topic": "접이식 전기 자전거"}, stream_mode="updates"):
    for node, upd in step.items():
        print(node, "→", upd)

## Step 05 · Command · Send

In [ ]:
# Command — 상태 갱신(update)과 다음 노드 이동(goto)을 노드 안에서 한 번에
from langgraph.types import Command

class TicketState(TypedDict):
    text: str
    priority: str
    handled_by: str

In [ ]:
def triage(state: TicketState) -> Command[Literal["urgent_team", "normal_team"]]:
    urgent = any(k in state["text"] for k in ["장애", "결제 실패", "접속 불가"])
    return Command(
        update={"priority": "high" if urgent else "normal"},
        goto="urgent_team" if urgent else "normal_team",
    )

def urgent_team(state: TicketState):
    return {"handled_by": "온콜 엔지니어"}

def normal_team(state: TicketState):
    return {"handled_by": "고객지원팀"}

In [ ]:
tb = StateGraph(TicketState)
tb.add_node("triage", triage)          # 조건부 엣지 없이 Command로 이동
tb.add_node("urgent_team", urgent_team)
tb.add_node("normal_team", normal_team)
tb.add_edge(START, "triage")
tb.add_edge("urgent_team", END)
tb.add_edge("normal_team", END)
ticket_graph = tb.compile()

In [ ]:
print(ticket_graph.invoke({"text": "결제 실패가 계속 발생합니다"}))

In [ ]:
print(ticket_graph.invoke({"text": "영수증 재발급 부탁드려요"}))

In [ ]:
# Send — 리스트 원소마다 같은 노드를 병렬 실행하고(Map) 결과를 모은다(Reduce)
from langgraph.types import Send

class ReviewsState(TypedDict):
    reviews: list[str]
    summaries: Annotated[list[str], operator.add]   # 병렬 결과를 누적
    report: str

class OneReview(TypedDict):
    review: str

In [ ]:
def fan_out(state: ReviewsState):
    return [Send("summarize_one", {"review": r}) for r in state["reviews"]]

def summarize_one(state: OneReview):
    s = llm.invoke(f"다음 리뷰의 핵심을 10자 이내로 요약: {state['review']}").content
    return {"summaries": [s]}

def make_report(state: ReviewsState):
    joined = "\n".join(f"- {s}" for s in state["summaries"])
    return {"report": llm.invoke(f"다음 리뷰 요약들을 보고 개선점 1가지를 제안하세요.\n{joined}").content}

In [ ]:
mb = StateGraph(ReviewsState)
mb.add_node("summarize_one", summarize_one)
mb.add_node("report", make_report)
mb.add_conditional_edges(START, fan_out, ["summarize_one"])
mb.add_edge("summarize_one", "report")
mb.add_edge("report", END)
mr_graph = mb.compile()

In [ ]:
res = mr_graph.invoke({"reviews": ["배송이 빨라요", "사이즈가 작게 나와요", "색이 사진과 달라요", "가성비 최고"]})
print(res["summaries"])
print(res["report"])

## Step 06 · 도구 에이전트

- ToolNode · tools_condition

In [ ]:
# `create_agent`가 내부에서 만드는 에이전트 루프를 그래프로 직접 구성한다.

In [ ]:
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

STOCK = {"러닝화": 12, "요가매트": 0, "텀블러": 40}

@tool
def check_stock(product: str) -> str:
    """상품명의 재고 수량을 조회합니다."""
    return f"{product} 재고: {STOCK.get(product, '없는 상품')}"

@tool
def restock_date(product: str) -> str:
    """품절 상품의 재입고 예정일을 조회합니다."""
    return {"요가매트": "10월 15일"}.get(product, "재입고 계획 없음")

In [ ]:
tools = [check_stock, restock_date]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def agent(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
ab = StateGraph(MessagesState)
ab.add_node("agent", agent)
ab.add_node("tools", ToolNode(tools))
ab.add_edge(START, "agent")
ab.add_conditional_edges("agent", tools_condition)   # tool_calls 있으면 "tools", 없으면 END
ab.add_edge("tools", "agent")
tool_agent = ab.compile()

In [ ]:
out = tool_agent.invoke({"messages": [("user", "요가매트 재고 있어? 없으면 언제 들어와?")]})
for m in out["messages"]:
    print(f"[{m.type}]", m.content or m.tool_calls)

## Step 07 · Persistence · Human-in-the-Loop

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
mem_agent = ab.compile(checkpointer=memory)

cfg = {"configurable": {"thread_id": "user-1"}}
mem_agent.invoke({"messages": [("user", "텀블러 재고 알려줘")]}, cfg)
out = mem_agent.invoke({"messages": [("user", "방금 물어본 상품 이름이 뭐였지?")]}, cfg)
print(out["messages"][-1].content)

other = {"configurable": {"thread_id": "user-2"}}
print(mem_agent.invoke({"messages": [("user", "방금 물어본 상품 이름이 뭐였지?")]}, other)["messages"][-1].content)

In [ ]:
# 저장된 상태 조회 · 체크포인트 이력
snap = mem_agent.get_state(cfg)
print("메시지 수:", len(snap.values["messages"]), "| 다음 노드:", snap.next)

history = list(mem_agent.get_state_history(cfg))
for h in history[:6]:
    print(f"step={h.metadata['step']:>2}  next={h.next}  msgs={len(h.values.get('messages', []))}")

In [ ]:
# Time Travel — 과거 체크포인트에서 다른 질문으로 다시 실행 (새 분기 생성)
first_turn_end = [h for h in history if not h.next][-1]   # 이력은 최신순 → 가장 오래된 "완료" 시점 = 첫 대화 종료
print("분기 시점 step:", first_turn_end.metadata["step"], "| 메시지 수:", len(first_turn_end.values["messages"]))
branch = mem_agent.invoke({"messages": [("user", "러닝화 재고는?")]}, first_turn_end.config)
print(branch["messages"][-1].content)

In [ ]:
# interrupt — 노드 실행 중 멈추고 사람의 결정을 받아 재개
from langgraph.types import interrupt

class RefundState(TypedDict):
    order_id: str
    amount: int
    approved: bool
    result: str

In [ ]:
def request_approval(state: RefundState):
    if state["amount"] < 100_000:                 # 소액은 자동 승인
        return {"approved": True}
    decision = interrupt({"question": "고액 환불 승인할까요?", "order_id": state["order_id"],
                          "amount": state["amount"]})
    return {"approved": decision == "approve"}

def execute(state: RefundState):
    return {"result": f"{state['order_id']} 환불 {'완료' if state['approved'] else '거절'}"}

In [ ]:
hb = StateGraph(RefundState)
hb.add_node("approval", request_approval)
hb.add_node("execute", execute)
hb.add_edge(START, "approval")
hb.add_edge("approval", "execute")
hb.add_edge("execute", END)
hitl = hb.compile(checkpointer=InMemorySaver())    # interrupt에는 checkpointer 필수

c = {"configurable": {"thread_id": "refund-77"}}
first = hitl.invoke({"order_id": "A-77", "amount": 250_000}, c)
print("중단:", first["__interrupt__"][0].value)
print("다음 노드:", hitl.get_state(c).next)

final = hitl.invoke(Command(resume="approve"), c)   # 사람의 결정으로 재개
print(final["result"])

## Step 08 · Streaming · Subgraph

In [ ]:
# stream_mode 비교
q = {"messages": [("user", "러닝화 재고 알려줘")]}

print("--- updates: 노드별 변경분")
for chunk in tool_agent.stream(q, stream_mode="updates"):
    print(list(chunk.keys()))

In [ ]:
print("--- values: 매 단계 전체 상태")
for chunk in tool_agent.stream(q, stream_mode="values"):
    print(len(chunk["messages"]), "개 메시지")

In [ ]:
print("--- messages: LLM 토큰")
for token, meta in tool_agent.stream({"messages": [("user", "LangGraph 장점 한 줄")]}, stream_mode="messages"):
    if token.content and meta["langgraph_node"] == "agent":
        print(token.content, end="", flush=True)

In [ ]:
# custom — 노드 안에서 진행 상황을 직접 보내기
from langgraph.config import get_stream_writer

class JobState(TypedDict):
    files: list[str]
    done: int

In [ ]:
def process(state: JobState):
    writer = get_stream_writer()
    for i, f in enumerate(state["files"], 1):
        writer({"progress": f"{i}/{len(state['files'])} {f} 처리 중"})
    return {"done": len(state["files"])}

In [ ]:
job = StateGraph(JobState).add_node("process", process).add_edge(START, "process").compile()

In [ ]:
for mode, chunk in job.stream({"files": ["a.pdf", "b.pdf", "c.pdf"]}, stream_mode=["custom", "updates"]):
    print(mode, chunk)

In [ ]:
# Subgraph — 컴파일한 그래프를 다른 그래프의 노드로 사용 (State 키를 공유)
class PipelineState(TypedDict):
    question: str
    category: str
    answer: str
    polished: str

In [ ]:
def polish(state: PipelineState):
    return {"polished": llm.invoke(f"다음 답변을 공손한 존댓말 한 문장으로 다듬으세요: {state['answer']}").content}

In [ ]:
pb = StateGraph(PipelineState)
pb.add_node("route", router)          # Step 04의 문의 라우터 그래프를 노드로
pb.add_node("polish", polish)
pb.add_edge(START, "route")
pb.add_edge("route", "polish")
pb.add_edge("polish", END)
pipeline = pb.compile()

In [ ]:
for ns, chunk in pipeline.stream({"question": "환불 어떻게 해요?"}, stream_mode="updates", subgraphs=True):
    print(ns or "(parent)", list(chunk.keys()))
print(pipeline.invoke({"question": "환불 어떻게 해요?"})["polished"])